In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('.').resolve().parent))

# ── Config ────────────────────────────────────────────────────────────────
from src.config_loader import ConfigLoader
cfg = ConfigLoader.get_instance('../config.yaml')

# ── Plot aesthetics ───────────────────────────────────────────────────────
PALETTE   = cfg.get('eda.palette', 'viridis')
FIGSIZE   = tuple(cfg.get('eda.figsize_default', [12, 6]))
CHURN_PAL = {'No Churn': '#4A90D9', 'Churn': '#E94B3C'}

sns.set_theme(style='whitegrid', palette=PALETTE, font_scale=1.05)
plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

print('✅ Setup complete')
print('Config loaded from:', cfg._config_path)

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────
raw_path = Path('..') / cfg.get('data.raw_path')
df_raw   = pd.read_csv(raw_path)

# Coerce TotalCharges (IBM dataset ships it as string)
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')

TARGET   = cfg.get('data.target_col')          # 'Churn'
ID_COL   = cfg.get('data.customer_id_col')     # 'customerID'
NUM_COLS = cfg.get('data.schema.numerical_cols')
CAT_COLS = cfg.get('data.schema.categorical_cols')

print(f'Dataset shape: {df_raw.shape}')
print(f'Target: {TARGET}  |  ID: {ID_COL}')
df_raw.head(3)

In [ ]:
# ── 2.1 Missing Values ────────────────────────────────────────────────────
null_counts = df_raw.isnull().sum()
null_pct    = (null_counts / len(df_raw) * 100).round(2)
null_df     = pd.DataFrame({'count': null_counts, 'pct': null_pct})
null_df     = null_df[null_df['count'] > 0].sort_values('pct', ascending=False)

print('=== Columns with Missing Values ===')
display(null_df)

fig, ax = plt.subplots(figsize=(10, 4))
null_heat = df_raw.isnull().sum().to_frame('nulls').T
sns.heatmap(null_heat, annot=True, fmt='d', cmap='Reds', linewidths=0.5, ax=ax)
ax.set_title('Null Count Heatmap (per column)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.2 Data Types & Unique Values ────────────────────────────────────────
dtype_df = pd.DataFrame({
    'dtype':   df_raw.dtypes,
    'n_unique': df_raw.nunique(),
    'sample':   df_raw.iloc[0],
})
display(dtype_df)

In [ ]:
# ── 2.3 Duplicate Check ───────────────────────────────────────────────────
n_dups = df_raw.duplicated().sum()
n_dup_ids = df_raw[ID_COL].duplicated().sum()
print(f'Duplicate rows        : {n_dups}')
print(f'Duplicate customerIDs : {n_dup_ids}')
print(f'Dataset is clean      : {n_dups == 0 and n_dup_ids == 0}')

In [ ]:
# ── 2.4 Statistical Summary ───────────────────────────────────────────────
df_raw[NUM_COLS].describe().round(2)

In [ ]:
# Map Churn to readable labels for plotting
df_plot = df_raw.copy()
df_plot['Churn_Label'] = df_plot[TARGET].map({'Yes': 'Churn', 'No': 'No Churn', 1: 'Churn', 0: 'No Churn'})

churn_counts = df_plot['Churn_Label'].value_counts()
churn_pct    = churn_counts / len(df_plot) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(churn_counts.index, churn_counts.values,
            color=[CHURN_PAL.get(k, '#888') for k in churn_counts.index],
            edgecolor='white', linewidth=1.5)
for i, (cnt, pct) in enumerate(zip(churn_counts.values, churn_pct.values)):
    axes[0].text(i, cnt + 20, f'{cnt:,}\n({pct:.1f}%)', ha='center', fontsize=12)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, churn_counts.max() * 1.15)

# Pie chart
axes[1].pie(churn_counts.values,
            labels=churn_counts.index,
            autopct='%1.1f%%',
            colors=[CHURN_PAL.get(k, '#888') for k in churn_counts.index],
            startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Ratio', fontsize=14, fontweight='bold')

plt.suptitle('Target Variable: Class Imbalance', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Churn rate: {churn_pct["Churn"]:.2f}%')
print(f'Imbalance ratio (majority:minority): {churn_pct["No Churn"]/churn_pct["Churn"]:.2f}:1')

In [ ]:
# ── 4.1 Numerical Distributions ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4A90D9', '#27AE60', '#E67E22']

for ax, col, color in zip(axes, NUM_COLS, colors):
    ax.hist(df_raw[col].dropna(), bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(df_raw[col].mean(),   color='red',    ls='--', lw=1.5, label=f'Mean={df_raw[col].mean():.1f}')
    ax.axvline(df_raw[col].median(), color='purple', ls=':',  lw=1.5, label=f'Median={df_raw[col].median():.1f}')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('Numerical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2 Categorical Distributions ─────────────────────────────────────────
cat_sample = ['Contract', 'PaymentMethod', 'InternetService', 'MultipleLines',
              'TechSupport', 'OnlineSecurity']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, cat_sample):
    vc = df_raw[col].value_counts()
    bars = ax.barh(vc.index, vc.values, color=sns.color_palette('viridis', len(vc)))
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f'{val:,}', va='center', fontsize=10)
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel('Count')

plt.suptitle('Key Categorical Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.1 Churn Rate by Contract Type ───────────────────────────────────────
# Create a binary churn column for analysis
df_raw['Churn_Binary'] = df_raw[TARGET].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})

contract_churn = df_raw.groupby('Contract')['Churn_Binary'].agg(['mean', 'count']).reset_index()
contract_churn['churn_rate'] = contract_churn['mean'] * 100
contract_churn = contract_churn.sort_values('churn_rate', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(contract_churn['Contract'], contract_churn['churn_rate'],
              color=['#E94B3C', '#E67E22', '#27AE60'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, contract_churn['churn_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, contract_churn['churn_rate'].max() * 1.2)
ax.axhline(df_raw['Churn_Binary'].mean()*100, color='navy', ls='--', lw=1.5, label='Overall avg')
ax.legend()
plt.tight_layout()
plt.show()
print('\nKey insight: Month-to-month customers churn at 4-5x the rate of two-year contract holders.')

In [ ]:
# ── 5.2 Churn Rate by Payment Method ──────────────────────────────────────
pay_churn = df_raw.groupby('PaymentMethod')['Churn_Binary'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colors_pay = ['#E94B3C' if r > 35 else '#E67E22' if r > 20 else '#27AE60' for r in pay_churn.values]
ax.barh(pay_churn.index, pay_churn.values, color=colors_pay, edgecolor='white')
for i, val in enumerate(pay_churn.values):
    ax.text(val + 0.3, i, f'{val:.1f}%', va='center', fontweight='bold')
ax.set_title('Churn Rate by Payment Method', fontsize=14, fontweight='bold')
ax.set_xlabel('Churn Rate (%)')
ax.axvline(df_raw['Churn_Binary'].mean()*100, color='navy', ls='--', lw=1.5, label='Overall avg')
ax.legend()
plt.tight_layout()
plt.show()
print('Insight: Electronic check users churn at ~2x the rate of auto-pay customers.')

In [ ]:
# ── 5.3 Tenure vs Churn — Boxplot & KDE ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churned     = df_raw[df_raw['Churn_Binary'] == 1]['tenure']
not_churned = df_raw[df_raw['Churn_Binary'] == 0]['tenure']

# Box
axes[0].boxplot([not_churned.dropna(), churned.dropna()],
                labels=['No Churn', 'Churn'],
                patch_artist=True,
                boxprops=dict(facecolor='#4A90D9', alpha=0.7),
                medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Tenure Distribution by Churn', fontweight='bold')
axes[0].set_ylabel('Tenure (months)')

# KDE
not_churned.plot.kde(ax=axes[1], label='No Churn', color='#4A90D9', lw=2)
churned.plot.kde(ax=axes[1], label='Churn', color='#E94B3C', lw=2)
axes[1].set_title('Tenure KDE by Churn Status', fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].legend()

plt.suptitle('Tenure vs Churn', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Churners median tenure    : {churned.median():.0f} months')
print(f'Non-churners median tenure: {not_churned.median():.0f} months')
print('Insight: Most churners leave within the first 12 months.')

In [ ]:
# ── 5.4 MonthlyCharges & TotalCharges by Churn ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['MonthlyCharges', 'TotalCharges']):
    data_churn    = df_raw[df_raw['Churn_Binary'] == 1][col].dropna()
    data_no_churn = df_raw[df_raw['Churn_Binary'] == 0][col].dropna()
    ax.violinplot([data_no_churn, data_churn], positions=[0, 1],
                  showmedians=True, showextrema=True)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['No Churn', 'Churn'])
    ax.set_title(f'{col} by Churn', fontweight='bold')
    ax.set_ylabel(f'{col} (USD)')

plt.suptitle('Charge Distributions by Churn Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.1 Correlation Heatmap ───────────────────────────────────────────────
corr_method = cfg.get('eda.correlation_method', 'spearman')
num_df = df_raw[NUM_COLS + ['Churn_Binary']].dropna()
corr   = num_df.corr(method=corr_method)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title(f'Correlation Heatmap ({corr_method.capitalize()})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

churn_corr = corr['Churn_Binary'].drop('Churn_Binary').sort_values(key=abs, ascending=False)
print('Correlation with Churn:')
print(churn_corr)

In [ ]:
# ── 6.2 Pair Plot ─────────────────────────────────────────────────────────
sample_n = min(cfg.get('eda.sample_n_for_plots', 2000), len(df_raw))
pair_df  = df_raw[NUM_COLS + ['Churn_Binary']].dropna().sample(sample_n, random_state=42)
pair_df['Churn_Label'] = pair_df['Churn_Binary'].map({0: 'No Churn', 1: 'Churn'})

g = sns.pairplot(pair_df, hue='Churn_Label', palette=CHURN_PAL,
                 vars=NUM_COLS, plot_kws={'alpha': 0.4, 's': 15},
                 diag_kind='kde')
g.fig.suptitle('Pair Plot — Numerical Features by Churn', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3 Contract x PaymentMethod Churn Heatmap ────────────────────────────
pivot = df_raw.pivot_table(values='Churn_Binary', index='Contract',
                            columns='PaymentMethod', aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, vmin=0, vmax=60)
ax.set_title('Churn Rate (%) — Contract × Payment Method', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Insight: Month-to-Month + Electronic Check = highest-risk combination (>40% churn).')

In [ ]:
# ── 7.1 IQR Box Plots ─────────────────────────────────────────────────────
outlier_cols = cfg.get('preprocessing.outlier.columns_to_check')
iqr_factor   = cfg.get('preprocessing.outlier.iqr_factor', 1.5)

fig, axes = plt.subplots(1, len(outlier_cols), figsize=(14, 5))
for ax, col in zip(axes, outlier_cols):
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr   = q3 - q1
    lower = q1 - iqr_factor * iqr
    upper = q3 + iqr_factor * iqr
    n_out = ((df_raw[col] < lower) | (df_raw[col] > upper)).sum()
    ax.boxplot(df_raw[col].dropna(), patch_artist=True,
               boxprops=dict(facecolor='#4A90D9', alpha=0.7),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col}\n({n_out} outliers)', fontweight='bold')
    ax.set_ylabel(col)

plt.suptitle(f'IQR Outlier Detection (factor={iqr_factor})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7.2 Z-Score Table ─────────────────────────────────────────────────────
z_thresh = cfg.get('preprocessing.outlier.zscore_threshold', 3.0)
z_scores = df_raw[outlier_cols].apply(stats.zscore, nan_policy='omit').abs()
outlier_mask = (z_scores > z_thresh).any(axis=1)
n_z_outliers = outlier_mask.sum()
print(f'Z-score outliers (|z| > {z_thresh}): {n_z_outliers} rows ({n_z_outliers/len(df_raw)*100:.2f}%)')

if n_z_outliers > 0:
    display(df_raw[outlier_mask][outlier_cols].describe().round(2))

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

K = cfg.get('eda.cluster_k', 3)

seg_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
seg_df   = df_raw[seg_cols + ['Churn_Binary']].dropna().copy()

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(seg_df[seg_cols])

km = KMeans(n_clusters=K, random_state=42, n_init='auto')
seg_df['Cluster'] = km.fit_predict(X_scaled)

# Profile
profile = seg_df.groupby('Cluster').agg(
    n_customers=('Churn_Binary', 'count'),
    churn_rate=('Churn_Binary', 'mean'),
    avg_tenure=('tenure', 'mean'),
    avg_monthly=('MonthlyCharges', 'mean'),
    avg_total=('TotalCharges', 'mean'),
).round(2)
profile['churn_rate'] = (profile['churn_rate'] * 100).round(1)
print('=== Cluster Profiles ===')
display(profile)

In [ ]:
# Scatter: tenure vs MonthlyCharges coloured by cluster
sample_seg = seg_df.sample(min(1500, len(seg_df)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter_params = dict(alpha=0.5, s=20, edgecolors='none')

# By cluster
sc1 = axes[0].scatter(sample_seg['tenure'], sample_seg['MonthlyCharges'],
                      c=sample_seg['Cluster'], cmap='tab10', **scatter_params)
axes[0].set_title('Clusters: Tenure vs MonthlyCharges', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('MonthlyCharges ($)')
plt.colorbar(sc1, ax=axes[0], label='Cluster')

# By churn status
sc2 = axes[1].scatter(sample_seg['tenure'], sample_seg['MonthlyCharges'],
                      c=sample_seg['Churn_Binary'], cmap='RdYlGn_r', **scatter_params)
axes[1].set_title('Churn Status: Tenure vs MonthlyCharges', fontweight='bold')
axes[1].set_xlabel('Tenure (months)')
axes[1].set_ylabel('MonthlyCharges ($)')
plt.colorbar(sc2, ax=axes[1], label='Churn (1=Yes)')

plt.suptitle('Customer Segmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Revenue-at-Risk Calculation ───────────────────────────────────────────
biz    = cfg.get_section('reporting')['business']
ARPU   = biz['avg_monthly_revenue_per_customer']    # $65/month
AVG_LT = biz['avg_customer_lifetime_months']         # 32 months
RET_COST = biz['retention_offer_cost']               # $50/customer
RET_RATE = biz['retention_success_rate']             # 35%

total_customers = len(df_raw)
n_churners      = df_raw['Churn_Binary'].sum()
churn_rate      = n_churners / total_customers

revenue_at_risk = n_churners * ARPU * AVG_LT
intervention_cost = n_churners * RET_COST
customers_saved   = n_churners * RET_RATE
revenue_saved     = customers_saved * ARPU * AVG_LT
net_benefit       = revenue_saved - intervention_cost
roi_pct           = (net_benefit / intervention_cost) * 100

print('╔══════════════════════════════════════════════════════════╗')
print('║           BUSINESS IMPACT ANALYSIS                       ║')
print('╠══════════════════════════════════════════════════════════╣')
print(f'║  Total Customers          : {total_customers:>8,}                   ║')
print(f'║  Identified Churners      : {int(n_churners):>8,} ({churn_rate*100:.1f}%)              ║')
print(f'║  Revenue at Risk          : ${revenue_at_risk:>10,.0f}                ║')
print(f'║  Intervention Cost        : ${intervention_cost:>10,.0f}                ║')
print(f'║  Customers Saved (35%)    : {int(customers_saved):>8,}                   ║')
print(f'║  Revenue Saved            : ${revenue_saved:>10,.0f}                ║')
print(f'║  Net Benefit              : ${net_benefit:>10,.0f}                ║')
print(f'║  ROI                      : {roi_pct:>8.1f}%                  ║')
print('╚══════════════════════════════════════════════════════════╝')

In [ ]:
# ── High-Risk Profile Identification ─────────────────────────────────────
high_risk_thresh = cfg.get('reporting.high_risk_threshold', 0.70)

# Define high-risk heuristically based on known churn drivers
df_raw['high_risk_flag'] = (
    (df_raw['Contract'] == 'Month-to-month') &
    (df_raw['PaymentMethod'] == 'Electronic check') &
    (df_raw['tenure'] <= 12)
).astype(int)

high_risk_churn_rate = df_raw[df_raw['high_risk_flag'] == 1]['Churn_Binary'].mean() * 100
n_high_risk = df_raw['high_risk_flag'].sum()

print(f'\n🚨 High-Risk Profile: Month-to-Month + Electronic Check + Tenure ≤12mo')
print(f'   N customers    : {n_high_risk:,}')
print(f'   Churn rate     : {high_risk_churn_rate:.1f}%  (vs {churn_rate*100:.1f}% overall)')
print(f'   Uplift         : {high_risk_churn_rate/(churn_rate*100):.1f}x average risk')

In [ ]:
# ── 9.3 Top 5 Key Findings Summary ───────────────────────────────────────
print('''
╔══════════════════════════════════════════════════════════════════╗
║               KEY BUSINESS FINDINGS                              ║
╠══════════════════════════════════════════════════════════════════╣
║ 1. CONTRACT TYPE is the #1 churn predictor.                      ║
║    Month-to-month customers churn at ~42% vs 11% (One year)      ║
║    and ~3% (Two year). Upselling to annual plans is high ROI.    ║
╠══════════════════════════════════════════════════════════════════╣
║ 2. EARLY TENURE is critical. ~50% of churners leave within       ║
║    the first 12 months. Onboarding & 30-day check-ins matter.   ║
╠══════════════════════════════════════════════════════════════════╣
║ 3. PAYMENT METHOD signals intent. Electronic check users         ║
║    churn at ~45% vs ~15% for auto-pay customers. Incentivise    ║
║    switching to bank transfer or credit card autopay.            ║
╠══════════════════════════════════════════════════════════════════╣
║ 4. HIGH MONTHLY CHARGES amplify churn risk. Customers paying    ║
║    >$80/mo with no long-term contract are most price-sensitive.  ║
╠══════════════════════════════════════════════════════════════════╣
║ 5. FIBER OPTIC customers churn more than DSL despite higher LTV. ║
║    Likely driven by price + competition. Bundle discounts help.  ║
╚══════════════════════════════════════════════════════════════════╝
''')